# MotherDuck Silver Layer Validation
Let's verify that the new `dbt-core` transformations actually work in the cloud MotherDuck database!

In [1]:
import duckdb
import pandas as pd
import os
from dotenv import load_dotenv

load_dotenv()
token = os.getenv('MOTHERDUCK_TOKEN')
db = os.getenv('MD_DATABASE', 'my_db')

con = duckdb.connect(f'md:{db}?token={token}')
print('Connected to MotherDuck!')

Connected to MotherDuck!


### 1. Bronze Layer Counts

In [2]:
con.execute("""
    SELECT 'rent_bronze' as table_name, count(*) as cnt FROM bronze.rent_bronze
    UNION ALL
    SELECT 'sale_bronze', count(*) FROM bronze.sale_bronze
""").df()

,table_name,cnt
0,rent_bronze,14655
1,sale_bronze,16002


### 2. Silver Identity (Unique Listings)

In [3]:
con.execute("""
    SELECT 
        mode,
        count(*) as total_listings,
        min(first_seen_at) as earliest_listing,
        max(last_seen_at) as latest_listing
    FROM silver.listing_identity
    GROUP BY mode
""").df()

,mode,total_listings,earliest_listing,latest_listing
0,sale,1618,2026-04-04 19:13:46.167800+02:00,2026-07-18 07:41:33.566005+02:00
1,rent,2190,2026-04-04 19:13:39.991741+02:00,2026-07-18 07:41:29.991680+02:00


### 3. Silver Versions (SCD Type 2)

In [4]:
con.execute("""
    SELECT 
        mode,
        count(*) as total_versions,
        count(distinct source_listing_id) as unique_listings,
        count(*) - count(distinct source_listing_id) as price_changes_recorded
    FROM silver.listing_versions
    GROUP BY mode
""").df()

,mode,total_versions,unique_listings,price_changes_recorded
0,rent,2550,2190,360
1,sale,1817,1618,199


### 4. Silver Current (Active State)

In [5]:
con.execute("""
    SELECT 
        title, 
        price_total, 
        city, 
        dbt_valid_from
    FROM silver.listing_current 
    ORDER BY dbt_valid_from DESC 
    LIMIT 5
""").df()

,title,price_total,city,dbt_valid_from
0,"54,5 m² | Duży salon | Oddzielna kuchnia | Balkon",667108.0,Kraków,2026-07-18 07:41:33.566005+02:00
1,NIŻSZA CENA - 2 pokoje - 52 m2 - ul. Kazimierz...,799000.0,Kraków,2026-07-18 07:41:33.566005+02:00
2,Dom wolnostojący w doskonałej lokalizacji,1590000.0,Kraków,2026-07-18 07:41:33.566005+02:00
3,Mieszkanie 2 pokojowe ul Bochenka,599900.0,Kraków,2026-07-18 07:41:33.566005+02:00
4,Mieszkanie 47 m² z klimatyzacją | Piasta Tower...,795000.0,Kraków,2026-07-18 07:41:33.566005+02:00


### 5. Price Insights (Krakow vs Suburbs)


Let's look at the median price per square meter for rent and sale, comparing Krakow with its surrounding municipalities.


In [ ]:
con.execute("""
    SELECT 
        mode, 
        IF(city='Kraków', 'Kraków', 'Suburbs') as area,
        count(*) as active_listings,
        median(price_per_sqm)::int as median_price_per_sqm,
        median(price_total)::int as median_total_price,
        median(area_sqm)::int as median_area_sqm
    FROM silver.listing_current
    WHERE price_per_sqm IS NOT NULL
    GROUP BY 1, 2
    ORDER BY mode, area
""").df()

,mode,area,active_listings,median_price_per_sqm,median_total_price,median_area_sqm
0,sale,Kraków,32,11097,788800,94
1,sale,Suburbs,83,9541,729000,78


### 6. Supply by Room Count in Krakow


What is the distribution of available properties by number of rooms?


In [ ]:
con.execute("""
    SELECT 
        mode,
        rooms,
        count(*) as active_listings
    FROM silver.listing_current
    WHERE city = 'Kraków' AND rooms IS NOT NULL
    GROUP BY 1, 2
    ORDER BY mode, rooms
""").df()

,mode,rooms,active_listings
0,rent,1.0,19
1,rent,2.0,5
2,rent,3.0,2
3,rent,4.0,2
4,rent,5.0,2
5,rent,6.0,1
6,rent,9.0,1
7,sale,2.0,7
8,sale,3.0,8
9,sale,4.0,1


### 7. Schema Description


Let's look at the structure of `silver.listing_current`.


In [ ]:
con.execute("""
DESCRIBE silver.listing_current
""").df()

,column_name,column_type,null,key,default,extra
0,dbt_scd_id,VARCHAR,YES,None,None,None
1,source,VARCHAR,YES,None,None,None
2,source_listing_id,VARCHAR,YES,None,None,None
3,mode,VARCHAR,YES,None,None,None
4,title,VARCHAR,YES,None,None,None
5,price_total,DOUBLE,YES,None,None,None
6,price_per_sqm,DOUBLE,YES,None,None,None
7,currency,VARCHAR,YES,None,None,None
8,area_sqm,DOUBLE,YES,None,None,None
9,rooms,DOUBLE,YES,None,None,None


### 8. Null Rates per Column


Identify which columns have the highest number of nulls in `silver.listing_current`.


In [ ]:
con.execute("""
SELECT
    count(*) as total_rows,
    count(title) as title_cnt,
    count(price_total) as price_total_cnt,
    count(price_per_sqm) as price_per_sqm_cnt,
    count(area_sqm) as area_sqm_cnt,
    count(rooms) as rooms_cnt,
    count(floor) as floor_cnt,
    count(city) as city_cnt,
    count(district) as district_cnt,
    count(seller_segment) as seller_segment_cnt
FROM silver.listing_current
""").df()

,total_rows,title_cnt,price_total_cnt,price_per_sqm_cnt,area_sqm_cnt,rooms_cnt,floor_cnt,city_cnt,district_cnt,seller_segment_cnt
0,3808,3808,3808,115,204,163,141,3808,3808,3808


### 9. Distributions of Numerical Columns


Summary statistics for area_sqm, price_total, and price_per_sqm across modes.


In [ ]:
con.execute("""
SELECT
    mode,
    count(area_sqm) as n_area,
    min(area_sqm) as min_area,
    quantile_cont(area_sqm, 0.25) as p25_area,
    median(area_sqm) as median_area,
    quantile_cont(area_sqm, 0.75) as p75_area,
    max(area_sqm) as max_area,
    min(price_total) as min_price,
    median(price_total) as median_price,
    max(price_total) as max_price
FROM silver.listing_current
GROUP BY mode
""").df()

,mode,n_area,min_area,p25_area,median_area,p75_area,max_area,min_price,median_price,max_price
0,rent,89,18.0,37.0,60.0,131.000,600.0,35.0,2300.0,27000.0
1,sale,115,23.0,50.0,81.0,125.755,600.0,70000.0,730000.0,25600000.0


### 10. Odd Values & Outliers


Checking for unusually low/high prices or areas.


In [ ]:
con.execute("""
SELECT 'Tiny Area (<10 sqm)' as issue, count(*) as cnt FROM silver.listing_current WHERE area_sqm < 10
UNION ALL
SELECT 'Massive Area (>1000 sqm)', count(*) FROM silver.listing_current WHERE area_sqm > 1000
UNION ALL
SELECT 'Extremely Low Total Price (<100 PLN)', count(*) FROM silver.listing_current WHERE price_total < 100
UNION ALL
SELECT 'Extremely High Total Price (>10M PLN)', count(*) FROM silver.listing_current WHERE price_total > 10000000
""").df()

,issue,cnt
0,Tiny Area (<10 sqm),0
1,Massive Area (>1000 sqm),0
2,Extremely Low Total Price (<100 PLN),1
3,Extremely High Total Price (>10M PLN),1


### 11. Exploring unnormalized `detail_params`


Let's see the keys in `detail_params` JSON in Bronze to identify what we can promote next.


In [ ]:
con.execute("""
WITH raw_data AS (
    SELECT raw_json::JSON as rj FROM bronze.rent_bronze
    UNION ALL
    SELECT raw_json::JSON FROM bronze.sale_bronze
),
param_keys AS (
    SELECT UNNEST(json_keys(json_extract(rj, '$.detail_params'))) as param_key
    FROM raw_data
)
SELECT param_key, count(*) as presence_count
FROM param_keys
GROUP BY param_key
ORDER BY presence_count DESC
LIMIT 15
""").df()

,param_key,presence_count
0,powierzchnia,518
1,rodzaj zabudowy,518
2,umeblowane,446
3,liczba pokoi,320
4,poziom,308
5,cena za m²,296
6,rynek,296
7,powierzchnia działki,194
8,liczba pięter,188
9,czynsz (dodatkowo),116


In [6]:
con.close()